# Disaster Tweet Classification

### College-Level Deep Learning Project

**Goal:** Predict whether a tweet is about a real disaster (`1`) or not (`0`).

Simple flow:

`Tweet → Clean Text → TF-IDF → Model → Prediction → Evaluation`


## 1. Import Libraries

We use pandas for data, scikit-learn for text processing and Logistic Regression, and TensorFlow/Keras for the simple neural network.


In [2]:
import re
import html
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

## 2. Load the Dataset

`train.csv` contains the tweets and their correct answers. `test.csv` contains tweets for final prediction.

`target = 1` means disaster, and `target = 0` means not disaster.


In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Training data:", train.shape)
print("Test data:", test.shape)

train.head()


Training data: (7612, 5)
Test data: (3263, 4)


,id,keyword,location,text,target
0,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
1,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
2,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
3,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
4,8,NaN,NaN,#RockyFire Update => California Hwy. 20 closed...,1


## 3. Clean the Tweets

We make the text easier for the model to understand by:
- converting text to lowercase
- removing links
- removing @mentions
- removing unnecessary symbols


In [4]:
# Platform filler words that say nothing about disasters
FILLER = r"\b(?:rt|link in bio|swipe up|see more|tag a friend|follow us)\b"

def clean_text(text):
    text = html.unescape(str(text))                                   # &amp; -> &
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)               # links
    text = re.sub(r"@\w+", " ", text)                                 # @mentions
    text = re.sub(r"#(\w+)",                                          # #HashTag -> hash tag
                  lambda m: " " + re.sub(r"(?<=[a-z0-9])(?=[A-Z])", " ", m.group(1)), text)
    text = text.lower()
    text = re.sub(FILLER, " ", text)                                  # RT, link in bio, ...
    text = re.sub(r"([a-z])\1{2,}", r"\1\1", text)                    # sooooo -> soo
    text = re.sub(r"[^a-z\s]", " ", text)                             # emojis, numbers, symbols
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["clean_text"] = train["text"].apply(clean_text)
test["clean_text"] = test["text"].apply(clean_text)

print("Original:", train.loc[0, "text"])
print("Cleaned :", train.loc[0, "clean_text"])

# Example of an Instagram-style post
example = "🚨 Wildfire near #LakeTahoe!!! Evacuate NOW &amp; stay safe 🔥🔥 @fireDept link in bio"
print()
print("Original:", example)
print("Cleaned :", clean_text(example))

Original: Forest fire near La Ronge Sask. Canada
Cleaned : forest fire near la ronge sask canada

Original: 🚨 Wildfire near #LakeTahoe!!! Evacuate NOW &amp; stay safe 🔥🔥 @fireDept link in bio
Cleaned : wildfire near lake tahoe evacuate now stay safe


## 4. Split the Data

We use 80% of the training data to train the model and 20% to evaluate it.


In [5]:
X = train["clean_text"]
y = train["target"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))


Training samples: 6089
Validation samples: 1523


## 5. Convert Text into Numbers using TF-IDF

Machine learning models cannot directly understand text. TF-IDF converts each tweet into numerical features.

We keep the 5,000 most useful words.


In [6]:
tfidf = TfidfVectorizer(max_features=5000, stop_words="english")

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(test["clean_text"])

print("TF-IDF shape:", X_train_tfidf.shape)


TF-IDF shape: (6089, 5000)


## 6. Model 1: Logistic Regression

This is our simple baseline model. We use it to compare the neural network with a traditional machine learning model.


In [7]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

lr_pred = lr_model.predict(X_val_tfidf)

print("Logistic Regression Accuracy:", accuracy_score(y_val, lr_pred))
print(classification_report(y_val, lr_pred))


Logistic Regression Accuracy: 0.8220617202889035
              precision    recall  f1-score   support

           0       0.81      0.90      0.85       869
           1       0.84      0.72      0.78       654

    accuracy                           0.82      1523
   macro avg       0.83      0.81      0.81      1523
weighted avg       0.82      0.82      0.82      1523



## 7. Model 2: Simple Neural Network

The neural network has:
- an input layer containing the TF-IDF features
- one hidden layer with ReLU
- one output neuron with Sigmoid

Sigmoid gives a value between 0 and 1, which is useful for binary classification.


In [8]:
nn_model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train_tfidf.shape[1],)),
    Dense(1, activation="sigmoid")
])

nn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

nn_model.summary()


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │       320,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,129 (1.22 MB)

 Trainable params: 320,129 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = nn_model.fit(
    X_train_tfidf.toarray(),
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val_tfidf.toarray(), y_val),
    verbose=1
)


Epoch 1/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.6373 - loss: 0.6605 - val_accuracy: 0.8148 - val_loss: 0.4893
Epoch 2/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8476 - loss: 0.4185 - val_accuracy: 0.8181 - val_loss: 0.4330
Epoch 3/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8827 - loss: 0.3059 - val_accuracy: 0.8063 - val_loss: 0.4480
Epoch 4/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9134 - loss: 0.2346 - val_accuracy: 0.7984 - val_loss: 0.4730
Epoch 5/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9269 - loss: 0.1997 - val_accuracy: 0.7873 - val_loss: 0.5076


## 8. Evaluate the Neural Network

In [10]:
nn_prob = nn_model.predict(X_val_tfidf.toarray(), verbose=0)
nn_pred = (nn_prob >= 0.5).astype(int).ravel()

print("Neural Network Accuracy:", accuracy_score(y_val, nn_pred))
print(classification_report(y_val, nn_pred))


Neural Network Accuracy: 0.7872619829284307
              precision    recall  f1-score   support

           0       0.81      0.82      0.81       869
           1       0.75      0.75      0.75       654

    accuracy                           0.79      1523
   macro avg       0.78      0.78      0.78      1523
weighted avg       0.79      0.79      0.79      1523



## 9. Compare the Models

We compare the accuracy of the simple machine learning model and the neural network.


In [11]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Neural Network"],
    "Accuracy": [
        accuracy_score(y_val, lr_pred),
        accuracy_score(y_val, nn_pred)
    ]
})

results


,Model,Accuracy
0,Logistic Regression,0.822062
1,Neural Network,0.787262


## 10. Predict the Test Tweets

The neural network is used to predict whether each test tweet is a disaster tweet.


In [12]:
test_prob = nn_model.predict(X_test_tfidf.toarray(), verbose=0)
test_pred = (test_prob >= 0.5).astype(int).ravel()

submission = pd.DataFrame({
    "id": test["id"],
    "target": test_pred
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully")
submission.head()


submission.csv created successfully


,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1


## 11. Try It on Instagram and Facebook Posts

The same cleaning, TF-IDF and neural network work on posts from any platform. `predict_posts` cleans the posts, converts them into numbers with the **same TF-IDF** learned earlier, and uses the neural network to predict.

`sample_posts.csv` contains a few example posts written for this demo (they are **not** part of the training data). The last two are deliberately tricky: they use disaster words in a joking way.

In [13]:
def predict_posts(posts, text_column="text"):
    """Predict disaster (1) or not (0) for posts from any platform."""
    cleaned = posts[text_column].astype(str).apply(clean_text)
    features = tfidf.transform(cleaned).toarray()
    prob = nn_model.predict(features, verbose=0).ravel()

    result = posts.copy()
    result["disaster_probability"] = prob.round(3)
    result["prediction"] = (prob >= 0.5).astype(int)

    # A post with only emojis or links has no words left to judge
    empty = cleaned == ""
    result.loc[empty, "disaster_probability"] = float("nan")
    result["prediction"] = result["prediction"].astype("Int64")
    result.loc[empty, "prediction"] = pd.NA
    return result

sample_posts = pd.read_csv("sample_posts.csv")
predict_posts(sample_posts)

FileNotFoundError: [Errno 2] No such file or directory: 'sample_posts.csv'

## 12. Use Your Own Posts

1. Save your Instagram or Facebook captions in a CSV file with a column named `text` (a `platform` column is optional).
2. Remove the `#` from the three lines below and run the cell.

The model learned from tweets, so treat its answers on other platforms as a good starting point, not a final verdict. Captions that are longer or use more slang can be harder. To improve it, label some of your own posts and add them to `train.csv` using the same `id`, `text`, `target` columns.

In [ ]:
# my_posts = pd.read_csv("my_posts.csv")           # needs a column called "text"
# my_results = predict_posts(my_posts)
# my_results.to_csv("my_predictions.csv", index=False)

## Conclusion

The project classifies social media posts (Twitter, Instagram, Facebook) as disaster or non-disaster posts.

TF-IDF converts text into numbers, Logistic Regression provides a simple baseline, and the neural network learns patterns from the numerical features.

### Main concepts demonstrated
- Text preprocessing (including hashtags and emojis)
- TF-IDF
- Binary classification
- Neural network
- ReLU activation
- Sigmoid activation
- Model training
- Model evaluation
- Final prediction
- Applying one model to several social media platforms